In [1]:
import os
import sys

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = "/scratch/shareddata/dldata/huggingface-hub-cache/hub" # Load local models
os.environ["TRANSFORMERS_OFFLINE"] = "1" 
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = f"{os.environ["WRKDIR"]}/.cache/huggingface" # Cache in work directory

sys.path.append(
    os.path.abspath(os.path.join(os.getcwd(), ".."))
)  # Add parent directory to path

from collections import defaultdict
import json
import pandas as pd
import transformers
from datasets import load_dataset

from utils.helpers import parse_output

from utils.constants import DEFAULT_AUGMENT_RESULT

from utils.prompts import (
    JUDGE_SYSTEM_PROMPT,
    JUDGE_TEMPLATE,
    AUGMENT_SYSTEM_PROMPT,
    AUGMENT_TEMPLATE,
)

In [2]:
DEFAULT_DATA = r"../../data/complete_dataset.csv"
DEFAULT_MODEL = "Qwen/Qwen2.5-14B-Instruct"

system_prompts = {
    "judge": JUDGE_SYSTEM_PROMPT,
    "augment": AUGMENT_SYSTEM_PROMPT
}

In [3]:
def make_prompt(row, task_type):
    match task_type:
        case "judge":
            return (
                JUDGE_TEMPLATE.replace("$THEME$", row["theme"])
                .replace("$TOPIC$", row["topic"])
                .replace("$CONCEPT$", row["concept"])
                .replace("$TEXT$", row["problemDescription"])
                .replace("$CODE$", row["exampleSolution"])
            )
        case "augment":
            return (
                AUGMENT_TEMPLATE.replace("$THEME$", row["theme"])
                .replace("$TOPIC$", row["topic"])
                .replace("$CONCEPT$", row["concept"])
                .replace("$TEXT$", row["problemDescription"])
                .replace("$CODE$", row["exampleSolution"])
            )
        case _:
            raise ValueError(f"Task type '{_}' not recognised as valid task type!")

In [4]:
task = "augment"
n_rows = 1
dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")

if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

dataset = dataset.map(lambda row: {"prompt": make_prompt(row, task)})

In [5]:
dataset.to_pandas().head(1)

,title,problemDescription,exampleSolution,starterCode,tests,theme,topic,concept,difficulty,id,The exercise description was clear (Yes/Partially/No),The exercise description matched the selected theme (Yes/Partially/No),The exercise description matched the selected topic (Yes/Partially/No),The exercise description matched the selected concept (Yes/No),Included concepts that were too advanced (Yes/No)\n,The exercise difficulty matched the selected difficulty (Too easy/Okay/Too difficult),Shallow vs deep personalization (Deep/Shallow/Unsure),Open field,prompt
0,Agatha Christie's Novel Ratings,"Agatha Christie, the famous novelist, has a ra...","{'code': ""import 'dart:io'; main() { print(...","{'code': ""import 'dart:io'; main() { }""}","{'testCode': ""import 'dart:async'; import 'pa...",literature,Agatha Christie,conditional statements,advanced,1.885152e+14,yes,yes,yes,yes,no,too easy,shallow,None,\nTheme: literature\nTopic: Agatha Christie\nC...


In [6]:
params = {
    "model": DEFAULT_MODEL,
    "device_map": 0,  # Force GPU
    "max_new_tokens": 1000,
    "temperature": 0.3,
}

pipeline = transformers.pipeline("text-generation", **params)
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
pipeline.model.config.pad_token_id = pipeline.model.config.eos_token_id

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


In [ ]:
results = defaultdict(list)
for prompt in dataset["prompt"]:
    output = pipeline(
        [
            {"role": "system", "content": system_prompts.get(task)},
            {"role": "user", "content": prompt},
        ],
        batch_size=1,
        return_full_text=False,
    )

    text = output[0]["generated_text"]

    parsed = parse_output(text)

    if parsed.get("Error") is not None:
        for k, v in DEFAULT_AUGMENT_RESULT.items():
            results[k].append(v)
        continue # Skip to next prompt

    keys = ["augmentedProblemDescription", "augmentedExampleSolution"] # Map to named lists
    for k in keys:
        results[k].append(parsed.get(k, "PARSE ERROR"))

for k, v in results.items():  # Add named lists as columns
    dataset = dataset.add_column(k, v)

print("done")

done


In [15]:
for i, row in dataset.to_pandas().iterrows():
    print(row["theme"])
    print(row["topic"])
    print(row["concept"])
    print(row["problemDescription"])
    print(row["exampleSolution"])

    print("")
    print(row["AugmentedDescription"])
    print(row["AugmentedSolution"])
    #print(row["theme"])

literature
Agatha Christie
conditional statements
Agatha Christie, the famous novelist, has a rating scale for her novels. The ratings are represented as numbers and are accompanied by the following textual descriptions:
<table>
<tr>
<th>Rating</th>
<th>Description</th>
</tr>
<tr>
<th>5</th>
<th>Masterpiece</th>
</tr>
<tr>
<th>4</th>
<th>Excellent</th>
</tr>
<tr>
<th>3</th>
<th>Good</th>
</tr>
<tr>
<th>2</th>
<th>Fair</th>
</tr>
<tr>
<th>1</th>
<th>Below Average</th>
</tr>
</table>
Write a program that asks the user for a number and prints the textual description related to that number. If the user enters any other number, the program should print the message <code>Invalid rating!</code>.

Below is an example of the expected operation of the program.

<pre>
What rating?
<b>&lt; 3</b>
Good
</pre>

Another example.

<pre>
What rating?
<b>&lt; 6</b>
Invalid rating!
</pre>
{'code': "import 'dart:io';  main() {   print('What rating?');    var rating = int.parse(stdin.readLineSync()!);    if

In [9]:
dataset.to_pandas().iloc[0]["problemDescription"]

'Agatha Christie, the famous novelist, has a rating scale for her novels. The ratings are represented as numbers and are accompanied by the following textual descriptions:\n<table>\n<tr>\n<th>Rating</th>\n<th>Description</th>\n</tr>\n<tr>\n<th>5</th>\n<th>Masterpiece</th>\n</tr>\n<tr>\n<th>4</th>\n<th>Excellent</th>\n</tr>\n<tr>\n<th>3</th>\n<th>Good</th>\n</tr>\n<tr>\n<th>2</th>\n<th>Fair</th>\n</tr>\n<tr>\n<th>1</th>\n<th>Below Average</th>\n</tr>\n</table>\nWrite a program that asks the user for a number and prints the textual description related to that number. If the user enters any other number, the program should print the message <code>Invalid rating!</code>.\n\nBelow is an example of the expected operation of the program.\n\n<pre>\nWhat rating?\n<b>&lt; 3</b>\nGood\n</pre>\n\nAnother example.\n\n<pre>\nWhat rating?\n<b>&lt; 6</b>\nInvalid rating!\n</pre>'